## Tokenization for arabic

In [2]:
import re
import unicodedata
import pandas as pd
import numpy as np
from collections import Counter
import nltk
from nltk.corpus import stopwords

# Download Arabic stopwords
nltk.download('stopwords')

# ==========================================
# STEP 1: Minimal Cleaning for Arabic Text
# ==========================================

# Precompiled patterns
URL_PATTERN = re.compile(r'\b(?:https?://|http://|www\.)\S+\b', re.IGNORECASE)
EMAIL_PATTERN = re.compile(r'\b[\w\.-]+@[\w\.-]+\.\w+\b')
MENTION_PATTERN = re.compile(r'@\w+')
NUMBER_PATTERN = re.compile(r'\d+')
WHITESPACE_PATTERN = re.compile(r'\s+')

# Arabic-specific patterns
ARABIC_DIACRITICS = re.compile(r'[\u064B-\u065F\u0670]')  # Tashkeel/diacritics
TATWEEL = re.compile(r'\u0640')  # Tatweel (ـ)

# Normalize Arabic characters
ARABIC_NORMALIZE_MAP = {
    'أ': 'ا', 'إ': 'ا', 'آ': 'ا', 'ٱ': 'ا',  # Alef variations
    'ة': 'ه',  # Taa Marbuta to Haa
    'ى': 'ي',  # Alef Maksura to Yaa
}

TRANSLATE_MAP = {
    "\u201c": '"', "\u201d": '"', "\u201e": '"', "\u201f": '"',
    "\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'",
    "\u2013": "-", "\u2014": "-", "\u2212": "-",
    "\u2026": "...",
    "\u00A0": " ",
}

def _normalize_unicode(s: str) -> str:
    s = unicodedata.normalize("NFC", s)
    return s.translate(str.maketrans(TRANSLATE_MAP))

def _remove_control_chars(s: str) -> str:
    return "".join(ch for ch in s if not unicodedata.category(ch).startswith("C"))

def _normalize_arabic(s: str) -> str:
    """Normalize Arabic-specific characters"""
    # Remove diacritics (tashkeel)
    s = ARABIC_DIACRITICS.sub('', s)
    # Remove tatweel
    s = TATWEEL.sub('', s)
    # Normalize Alef and other variations
    for old, new in ARABIC_NORMALIZE_MAP.items():
        s = s.replace(old, new)
    return s

def _replace_entities(s: str, replace_numbers: bool = True) -> str:
    s = EMAIL_PATTERN.sub("<EMAIL>", s)
    s = URL_PATTERN.sub("<URL>", s)
    s = MENTION_PATTERN.sub("<USER>", s)
    if replace_numbers:
        # Also replace Arabic-Indic digits (٠-٩)
        s = re.sub(r'[\u0660-\u0669]', '<NUM>', s)
        s = NUMBER_PATTERN.sub("<NUM>", s)
    return s

def _normalize_whitespace(s: str) -> str:
    s = WHITESPACE_PATTERN.sub(" ", s)
    return s.strip()

def clean_arabic_text(text: str, replace_numbers: bool = True) -> str:
    """
    Minimal cleaning for Arabic text:
    - Keep punctuation and emojis
    - Replace entities: URLs, emails, mentions, numbers
    - Normalize Arabic characters (remove diacritics, normalize Alef variants)
    - Remove control characters
    - Normalize whitespace
    """
    if not isinstance(text, str):
        return ""
    s = _normalize_unicode(text)
    s = _remove_control_chars(s)
    s = _normalize_arabic(s)
    s = _replace_entities(s, replace_numbers=replace_numbers)
    s = _normalize_whitespace(s)
    return s

def apply_arabic_cleaning(df: pd.DataFrame, text_col: str = "tweet") -> pd.DataFrame:
    """Apply Arabic cleaning to DataFrame"""
    df["text"] = df[text_col].apply(clean_arabic_text)
    return df


# ==========================================
# STEP 2: Tokenization for Arabic
# ==========================================

def preprocess_arabic_string(s):
    """
    Preprocess Arabic string:
    - Remove punctuation
    - Remove extra whitespace
    - Remove digits
    - Keep only Arabic letters
    """
    # Keep only Arabic letters, spaces
    s = re.sub(r'[^\u0600-\u06FF\s]', '', s)
    # Replace all runs of whitespaces with single space
    s = re.sub(r"\s+", ' ', s)
    return s.strip()


def tokenize_arabic(x_train, y_train, x_val, y_val, vocab_size=1000):
    """
    Tokenize Arabic text for training and validation sets

    Args:
        x_train: Training texts (list of strings)
        y_train: Training labels
        x_val: Validation texts (list of strings)
        y_val: Validation labels
        vocab_size: Maximum vocabulary size (default: 1000)

    Returns:
        final_list_train: List of token ID sequences for training
        encoded_train: Encoded training labels
        final_list_test: List of token ID sequences for validation
        encoded_test: Encoded validation labels
        onehot_dict: Dictionary mapping words to IDs
    """
    word_list = []

    # Load Arabic stopwords
    try:
        stop_words = set(stopwords.words('arabic'))
    except:
        print("Arabic stopwords not found. Using empty set.")
        stop_words = set()

    # Build vocabulary from training data
    for sent in x_train:
        if not isinstance(sent, str):
            continue
        for word in sent.split():
            word = preprocess_arabic_string(word)
            if word and word not in stop_words and len(word) > 1:
                word_list.append(word)

    # Count word frequencies
    corpus = Counter(word_list)
    # Get top N most common words
    corpus_ = sorted(corpus, key=corpus.get, reverse=True)[:vocab_size]
    # Create word-to-id mapping (1-indexed)
    onehot_dict = {w: i+1 for i, w in enumerate(corpus_)}

    print(f"Vocabulary size: {len(onehot_dict)}")
    print(f"Most common words: {corpus_[:10]}")

    # Tokenize training data
    final_list_train = []
    for sent in x_train:
        if not isinstance(sent, str):
            final_list_train.append([])
            continue
        tokens = [onehot_dict[preprocess_arabic_string(word)]
                 for word in sent.split()
                 if preprocess_arabic_string(word) in onehot_dict.keys()]
        final_list_train.append(tokens)

    # Tokenize validation data
    final_list_test = []
    for sent in x_val:
        if not isinstance(sent, str):
            final_list_test.append([])
            continue
        tokens = [onehot_dict[preprocess_arabic_string(word)]
                 for word in sent.split()
                 if preprocess_arabic_string(word) in onehot_dict.keys()]
        final_list_test.append(tokens)

    encoded_train = np.array(y_train)
    encoded_test = np.array(y_val)

    return final_list_train, encoded_train, final_list_test, encoded_test, onehot_dict


# ==========================================
# EXAMPLE USAGE
# ==========================================

if __name__ == "__main__":
    # Example with your data structure
    # Assuming you have a DataFrame 'df' with columns: tweet, label, etc.

    # Sample data based on your screenshot
    sample_data = {
        'tweet': [
            'والله شايفينك الناس كلها صابر يا عبد',
            'أكيد اكتتاب ي زينه شاء الله الحين',
            'والله انا ما انا عارف اكتتاب ولا انطلاء ولا',
        ],
        'label': [1, 1, 1]
    }

    df = pd.DataFrame(sample_data)

    # Step 1: Apply cleaning
    df_clean = apply_arabic_cleaning(df.copy(), text_col='tweet')
    print("Cleaned text:")
    print(df_clean['text'].head())
    print()

    # Step 2: Split data
    from sklearn.model_selection import train_test_split
    x_train, x_val, y_train, y_val = train_test_split(
        df_clean['text'].values,
        df_clean['label'].values,
        test_size=0.2,
        random_state=42
    )

    # Step 3: Tokenize
    final_list_train, encoded_train, final_list_test, encoded_test, vocab = tokenize_arabic(
        x_train, y_train, x_val, y_val, vocab_size=1000
    )

    print(f"Training sequences: {len(final_list_train)}")
    print(f"Validation sequences: {len(final_list_test)}")
    print(f"Sample tokenized sequence: {final_list_train[0]}")

Cleaned text:
0           والله شايفينك الناس كلها صابر يا عبد
1              اكيد اكتتاب ي زينه شاء الله الحين
2    والله انا ما انا عارف اكتتاب ولا انطلاء ولا
Name: text, dtype: object

Vocabulary size: 10
Most common words: ['اكتتاب', 'انا', 'اكيد', 'زينه', 'شاء', 'الله', 'الحين', 'والله', 'عارف', 'انطلاء']
Training sequences: 2
Validation sequences: 1
Sample tokenized sequence: [3, 1, 4, 5, 6, 7]


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
# Load your full dataset

df = pd.read_excel('Arabic_Depression_15.000_Tweets_Annotated.xlsx')
# Clean the text
df = apply_arabic_cleaning(df, text_col='tweet')

# Split into train/test
x_train, x_test, y_train, y_test = train_test_split(
    df['text'].values,
    df['label'].values,
    test_size=0.2,
    random_state=42
)

# Tokenize
final_list_train, encoded_train, final_list_test, encoded_test, vocab = tokenize_arabic(
    x_train, y_train, x_test, y_test, vocab_size=5000
)

# Now you can use with your SGNS training code

Vocabulary size: 5000
Most common words: ['اكتئاب', 'انا', 'علي', 'ان', 'سعيده', 'الله', 'سعاده', 'الي', 'حزن', 'متفائل']


# Padding

In [18]:
def padding_(sentences, seq_len):
    features = np.zeros((len(sentences), seq_len),dtype=int)
    for ii, review in enumerate(sentences):
        if len(review) != 0:
            features[ii, -len(review):] = np.array(review)[:seq_len]
    return features

# 1️ Split data
x_train, x_test, y_train, y_test = train_test_split(
    df["text"],    # features
    df["label"],   # correct label column
    test_size=0.2,
    random_state=42
)
# x_train=merged["text"].where(merged["split"] == "train"))
# x_test=merged.where(merged["split"] == "test").drop(["source","split"], axis=1)
# y_train=merged.where(merged["split"] == "train").drop(["source","split"], axis=1)


# 2️ Tokenize
final_list_train, encoded_train, final_list_test, encoded_test, vocab =  tokenize_arabic(
    x_train, y_train, x_test, y_test
)

# 3️ Pad sequences
x_train_pad = padding_(final_list_train, seq_len=50)
x_test_pad  = padding_(final_list_test, seq_len=50)

# 4️ Labels
y_train = encoded_train
y_test  = encoded_test

# Print the first 5 sequences
print("First 5 padded sequences:")
for i in range(5):
    print(x_train_pad[i])

# Print the shape of the padded training set
print("x_train_pad shape:", x_train_pad.shape)


Vocabulary size: 1000
Most common words: ['اكتئاب', 'انا', 'علي', 'ان', 'سعيده', 'الله', 'سعاده', 'الي', 'حزن', 'متفائل']
First 5 padded sequences:
[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0 206   2 393  85 635 102 405]
[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0 867 227   6   2]
[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0  79  66 140 199]
[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   6 379 457   7 338  68  22 933]
[  0   0   0   0   0   0   0   0   0

In [19]:
from torch.utils.data import TensorDataset, DataLoader
import torch as torch
# create Tensor datasets
train_data = TensorDataset(torch.from_numpy(x_train_pad), torch.from_numpy(y_train))
valid_data = TensorDataset(torch.from_numpy(x_test_pad), torch.from_numpy(y_test))


# dataloaders
batch_size = 50


# make sure to SHUFFLE your data
train_loader = DataLoader(train_data, shuffle=True, batch_size=batch_size)
valid_loader = DataLoader(valid_data, shuffle=True, batch_size=batch_size)


# obtain one batch of training data
dataiter = iter(train_loader)
sample_x, sample_y = next(dataiter)


print('Sample input size: ', sample_x.size()) # batch_size, seq_length
print('Sample input: \n', sample_x)
print('Sample output: \n', sample_y)

Sample input size:  torch.Size([50, 50])
Sample input: 
 tensor([[  0,   0,   0,  ..., 539, 638, 383],
        [  0,   0,   0,  ...,   0,   7,  41],
        [  0,   0,   0,  ..., 166,  29,  12],
        ...,
        [  0,   0,   0,  ..., 167, 744,   6],
        [  0,   0,   0,  ..., 532, 664, 131],
        [  0,   0,   0,  ...,  67, 740, 281]], dtype=torch.int32)
Sample output: 
 tensor([1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0,
        1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1,
        0, 0])


# trained word embeddings (we will use it as an input for the RNN LSTM or GRU )


In [22]:
import math
import random
from collections import Counter
from typing import List, Tuple, Dict, Optional

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import re


# =========================================
# Basic SGNS for your tokenize() outputs
# - final_list_train: List[List[int]] of 1-based token ids
# - onehot_dict: Dict[word -> id] with ids in [1..V]
# Adds robust validation/coercion for sequences to avoid TypeError.
# =========================================

def _coerce_int_token(tok) -> Optional[int]:
    # Accept Python/NumPy ints
    if isinstance(tok, (int, np.integer)):
        return int(tok)
    # Accept numeric strings like "123"
    if isinstance(tok, str):
        tok = tok.strip()
        if tok.isdigit():
            return int(tok)
        return None
    return None


def _coerce_and_validate_id_seqs(
    seqs_raw,
    vocab_size: int,
    max_report: int = 3,
) -> List[List[int]]:
    """
    Ensure we have List[List[int]] with ids in [1..vocab_size].
    - If a sequence is a list/array/tuple: keep only int-like tokens in range.
    - If a sequence is a single int-like token: wrap as a length-1 list.
    - If a sequence is a string of digits (e.g., '1 2 3' or '1,2,3'): parse digits.
    - Otherwise: raise with a helpful message.
    """
    cleaned: List[List[int]] = []
    bad_examples = []
    digit_re = re.compile(r"\d+")

    for s in seqs_raw:
        seq_ids: List[int] = []

        if isinstance(s, (list, tuple, np.ndarray)):
            for t in s:
                v = _coerce_int_token(t)
                if v is not None and 1 <= v <= vocab_size:
                    seq_ids.append(v)

        elif isinstance(s, (int, np.integer)):
            v = int(s)
            if 1 <= v <= vocab_size:
                seq_ids = [v]

        elif isinstance(s, str):
            # Try to parse numeric IDs from string
            nums = [int(m) for m in digit_re.findall(s)]
            seq_ids = [v for v in nums if 1 <= v <= vocab_size]

        else:
            bad_examples.append((type(s).__name__, s))
            continue

        if len(seq_ids) >= 2:
            cleaned.append(seq_ids)
        elif seq_ids:
            # Keep singletons? Usually not useful for skip-gram; drop to avoid empty batches.
            # You can keep them by changing this branch to cleaned.append(seq_ids).
            pass
        else:
            bad_examples.append((type(s).__name__, s))

    if not cleaned:
        sample = bad_examples[:max_report]
        raise TypeError(
            "final_list_train must be a list of integer-ID sequences (ids in [1..V]). "
            f"No valid sequences could be formed. Examples of invalid entries: {sample}"
        )

    if bad_examples:
        print(f"[warn] Skipped {len(bad_examples)} invalid/empty sequences while coercing to id lists.")

    return cleaned


def _build_token_freq(seqs: List[List[int]], vocab_size: int) -> np.ndarray:
    """Count token frequencies from 1-based index sequences."""
    freqs = np.zeros(vocab_size + 1, dtype=np.int64)  # +1 so index 0 remains unused
    for s in seqs:
        for t in s:
            if 1 <= t <= vocab_size:
                freqs[t] += 1
    return freqs


def _generate_skipgram_pairs(
    seqs: List[List[int]],
    window_size: int,
) -> List[Tuple[int, int]]:
    """Produce (center, context) positive pairs from 1-based token id sequences."""
    pairs: List[Tuple[int, int]] = []
    for seq in seqs:
        L = len(seq)
        if L < 2:
            continue
        for i, center in enumerate(seq):
            if center <= 0:
                continue
            left = max(0, i - window_size)
            right = min(L, i + window_size + 1)
            for j in range(left, right):
                if j == i:
                    continue
                ctx = seq[j]
                if ctx <= 0:
                    continue
                pairs.append((center, ctx))
    return pairs


class SkipGramPairsDS(Dataset):
    def __init__(self, pairs: List[Tuple[int, int]]):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        c, ctx = self.pairs[idx]
        return torch.tensor(c, dtype=torch.long), torch.tensor(ctx, dtype=torch.long)


def _unigram_noise(freqs: np.ndarray, power: float = 0.75) -> torch.Tensor:
    """
    Build noise distribution Pn(w) ∝ f(w)^power for indices [0..V], with prob[0]=0 (unused).
    """
    probs = freqs.astype(np.float64) ** power
    probs[0] = 0.0  # never sample index 0
    Z = probs.sum()
    if Z == 0:
        # fallback to uniform over 1..V
        probs[:] = 0.0
        probs[1:] = 1.0
        Z = probs.sum()
    probs /= Z
    return torch.tensor(probs, dtype=torch.float32)


class SGNS(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int):
        super().__init__()
        # +1 so row 0 stays unused; your ids are 1..vocab_size
        self.in_embed = nn.Embedding(vocab_size + 1, embed_dim)
        self.out_embed = nn.Embedding(vocab_size + 1, embed_dim)
        self.embed_dim = embed_dim
        self._init_weights()

    def _init_weights(self):
        bound = 0.5 / self.embed_dim
        nn.init.uniform_(self.in_embed.weight, -bound, bound)
        nn.init.zeros_(self.out_embed.weight)

    def forward(self, center: torch.Tensor, pos_ctx: torch.Tensor, neg_ctx: torch.Tensor) -> torch.Tensor:
        """
        center: [B] longs in [1..V]
        pos_ctx: [B] longs in [1..V]
        neg_ctx: [B, K] longs in [1..V]
        """
        v = self.in_embed(center)        # [B, D]
        u_pos = self.out_embed(pos_ctx)  # [B, D]
        u_neg = self.out_embed(neg_ctx)  # [B, K, D]

        # Positive term: log sigma(v · u_pos)
        pos_score = (v * u_pos).sum(dim=1)                  # [B]
        pos_loss = torch.log(torch.sigmoid(pos_score) + 1e-12)

        # Negative term: sum log sigma(-v · u_neg)
        neg_score = torch.bmm(u_neg.neg(), v.unsqueeze(2)).squeeze(2)  # [B, K]
        neg_loss = torch.log(torch.sigmoid(neg_score) + 1e-12).sum(dim=1)

        loss = -(pos_loss + neg_loss).mean()
        return loss


@torch.no_grad()
def extract_embedding_matrix(model: SGNS) -> np.ndarray:
    """
    Return the input embedding matrix including the unused row 0.
    Shape: [vocab_size+1, embed_dim]; your words are at rows 1..V.
    """
    return model.in_embed.weight.detach().cpu().numpy()


def train_sgns_for_tokenize_outputs(
    final_list_train,  # allow any; we coerce inside
    onehot_dict: Dict[str, int],
    embed_dim: int = 100,
    window_size: int = 2,
    num_negatives: int = 5,
    min_count: int = 1,            # your tokenize already prunes; keep 1
    batch_size: int = 4096,
    epochs: int = 2,
    lr: float = 0.01,
    device: Optional[str] = None,
    seed: int = 42,
) -> Tuple[np.ndarray, int]:
    """
    Train Word2Vec (Skip-gram with Negative Sampling) directly on your tokenize() outputs.

    Returns:
      - embedding_matrix: np.ndarray of shape [V+1, embed_dim] (row 0 unused)
      - vocab_size: V (so you can set nn.Embedding(V+1, embed_dim, padding_idx=0))
    """

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    vocab_size = max(onehot_dict.values())  # ids are 1..V

    # Coerce and validate sequences → List[List[int]] with ids in [1..V]
    final_list_train = _coerce_and_validate_id_seqs(final_list_train, vocab_size)

    # Build frequencies
    freqs = _build_token_freq(final_list_train, vocab_size)
    if min_count > 1:
        mask = (np.arange(freqs.size) > 0) & (freqs < min_count)
        freqs[mask] = 0

    # Positive pairs
    pairs = _generate_skipgram_pairs(final_list_train, window_size=window_size)
    if len(pairs) == 0:
        raise ValueError("No skip-gram pairs generated after coercion. Check your inputs.")

    ds = SkipGramPairsDS(pairs)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=False)

    # Negative sampling distribution
    noise = _unigram_noise(freqs, power=0.75)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    noise = noise.to(device)

    # Model + optimizer
    model = SGNS(vocab_size=vocab_size, embed_dim=embed_dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    # Train
    model.train()
    for ep in range(1, epochs + 1):
        total, steps = 0.0, 0
        for center, pos_ctx in dl:
            center = center.to(device)
            pos_ctx = pos_ctx.to(device)

            with torch.no_grad():
                neg = torch.multinomial(noise, num_samples=num_negatives * center.size(0), replacement=True)
                neg = neg.view(center.size(0), num_negatives)  # [B, K]

            opt.zero_grad(set_to_none=True)
            loss = model(center, pos_ctx, neg)
            loss.backward()
            opt.step()

            total += loss.item()
            steps += 1
        print(f"Epoch {ep}/{epochs} - avg loss: {total / max(1, steps):.4f}")

    emb = extract_embedding_matrix(model)  # [V+1, D], row 0 unused
    return emb, vocab_size

final_list_train, encoded_train, final_list_test, encoded_test, vocab = tokenize_arabic(
    x_train, y_train, x_test, y_test
)
emb, V = train_sgns_for_tokenize_outputs(
        final_list_train=final_list_train,
        onehot_dict=vocab,
        embed_dim=50,
        window_size=2,
        num_negatives=5,
        batch_size=8,
        epochs=5,
        lr=0.01,
    )
print(emb.shape)

Vocabulary size: 1000
Most common words: ['اكتئاب', 'انا', 'علي', 'ان', 'سعيده', 'الله', 'سعاده', 'الي', 'حزن', 'متفائل']
[warn] Skipped 111 invalid/empty sequences while coercing to id lists.
Epoch 1/5 - avg loss: 2.8849
Epoch 2/5 - avg loss: 2.8839
Epoch 3/5 - avg loss: 2.8926
Epoch 4/5 - avg loss: 2.8992
Epoch 5/5 - avg loss: 2.8947
(1001, 50)


. Arabic Feature Selection Module (arabic_feature_selection)

Chi-square test: Select words most correlated with labels
Mutual Information: Capture non-linear dependencies
Dimensionality Reduction: PCA and TruncatedSVD
Embedding Pooling: Mean, max, sum, and weighted pooling
TF-IDF Weighted Embeddings: Combine importance scores with semantics

In [23]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.feature_selection import chi2, mutual_info_classif, SelectKBest
from sklearn.decomposition import PCA, TruncatedSVD
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')


# ==========================================
# 1. STATISTICAL FEATURE SELECTION
# ==========================================

class ArabicFeatureSelector:
    """
    Statistical feature selection for Arabic text using:
    - Chi-square test
    - Mutual Information
    """

    def __init__(self, method='chi2', k=500):
        """
        Args:
            method: 'chi2' or 'mutual_info'
            k: number of top features to select
        """
        self.method = method
        self.k = k
        self.selector = None
        self.vectorizer = None
        self.selected_features = None

    def fit_transform(self, texts: List[str], labels: np.ndarray) -> np.ndarray:
        """
        Fit selector on training data and transform

        Args:
            texts: List of text strings
            labels: Array of labels

        Returns:
            Selected features matrix
        """
        # Create TF-IDF features
        self.vectorizer = TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),  # unigrams and bigrams
            min_df=2
        )
        X = self.vectorizer.fit_transform(texts)

        # Select scoring function
        if self.method == 'chi2':
            score_func = chi2
        elif self.method == 'mutual_info':
            score_func = mutual_info_classif
        else:
            raise ValueError(f"Unknown method: {self.method}")

        # Fit selector
        self.selector = SelectKBest(score_func=score_func, k=min(self.k, X.shape[1]))
        X_selected = self.selector.fit_transform(X, labels)

        # Get selected feature names
        feature_names = np.array(self.vectorizer.get_feature_names_out())
        selected_mask = self.selector.get_support()
        self.selected_features = feature_names[selected_mask]

        print(f"✓ Selected {X_selected.shape[1]} features using {self.method}")
        print(f"  Original features: {X.shape[1]}")
        print(f"  Top 10 features: {list(self.selected_features[:10])}")

        return X_selected

    def transform(self, texts: List[str]) -> np.ndarray:
        """Transform new texts using fitted selector"""
        if self.vectorizer is None or self.selector is None:
            raise ValueError("Must call fit_transform first")

        X = self.vectorizer.transform(texts)
        X_selected = self.selector.transform(X)
        return X_selected

    def get_feature_scores(self, top_n=20) -> pd.DataFrame:
        """Get scores for top N features"""
        if self.selector is None:
            raise ValueError("Must call fit_transform first")

        feature_names = np.array(self.vectorizer.get_feature_names_out())
        scores = self.selector.scores_
        selected_mask = self.selector.get_support()

        # Create DataFrame with scores
        df = pd.DataFrame({
            'feature': feature_names[selected_mask],
            'score': scores[selected_mask]
        })
        df = df.sort_values('score', ascending=False).head(top_n)
        return df


# ==========================================
# 2. DIMENSIONALITY REDUCTION
# ==========================================

class DimensionalityReducer:
    """
    Dimensionality reduction using PCA or TruncatedSVD
    """

    def __init__(self, method='pca', n_components=50):
        """
        Args:
            method: 'pca' or 'svd'
            n_components: number of components to keep
        """
        self.method = method
        self.n_components = n_components
        self.reducer = None

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        """
        Fit reducer and transform data

        Args:
            X: Feature matrix (dense or sparse)

        Returns:
            Reduced feature matrix
        """
        if self.method == 'pca':
            # PCA requires dense matrix
            if hasattr(X, 'toarray'):
                X = X.toarray()
            self.reducer = PCA(n_components=self.n_components, random_state=42)
        elif self.method == 'svd':
            # TruncatedSVD works with sparse matrices
            self.reducer = TruncatedSVD(n_components=self.n_components, random_state=42)
        else:
            raise ValueError(f"Unknown method: {self.method}")

        X_reduced = self.reducer.fit_transform(X)

        # Calculate variance explained
        if hasattr(self.reducer, 'explained_variance_ratio_'):
            var_explained = self.reducer.explained_variance_ratio_.sum()
            print(f"✓ Reduced to {self.n_components} dimensions using {self.method.upper()}")
            print(f"  Variance explained: {var_explained:.2%}")
        else:
            print(f"✓ Reduced to {self.n_components} dimensions using {self.method.upper()}")

        return X_reduced

    def transform(self, X: np.ndarray) -> np.ndarray:
        """Transform new data using fitted reducer"""
        if self.reducer is None:
            raise ValueError("Must call fit_transform first")

        if self.method == 'pca' and hasattr(X, 'toarray'):
            X = X.toarray()

        return self.reducer.transform(X)


# ==========================================
# 3. EMBEDDING-BASED FEATURE EXTRACTION
# ==========================================

class EmbeddingFeatureExtractor:
    """
    Extract features from word embeddings using various pooling strategies
    """

    def __init__(self, embedding_matrix: np.ndarray, vocab: Dict[str, int],
                 pooling='mean', use_pca=False, pca_dims=None):
        """
        Args:
            embedding_matrix: Shape [vocab_size+1, embed_dim], row 0 unused
            vocab: Dictionary mapping words to IDs (1-indexed)
            pooling: 'mean', 'max', 'sum', or 'weighted'
            use_pca: Whether to apply PCA after pooling
            pca_dims: Number of PCA dimensions (if use_pca=True)
        """
        self.embedding_matrix = embedding_matrix
        self.vocab = vocab
        self.pooling = pooling
        self.use_pca = use_pca
        self.pca_dims = pca_dims
        self.pca = None
        self.embed_dim = embedding_matrix.shape[1]

    def _pool_embeddings(self, token_ids: List[int], weights: Optional[List[float]] = None) -> np.ndarray:
        """
        Pool embeddings for a sequence of token IDs

        Args:
            token_ids: List of token IDs (1-indexed)
            weights: Optional weights for weighted pooling

        Returns:
            Pooled embedding vector
        """
        if len(token_ids) == 0:
            return np.zeros(self.embed_dim)

        # Get embeddings for all tokens
        embeddings = self.embedding_matrix[token_ids]  # [seq_len, embed_dim]

        if self.pooling == 'mean':
            return embeddings.mean(axis=0)
        elif self.pooling == 'max':
            return embeddings.max(axis=0)
        elif self.pooling == 'sum':
            return embeddings.sum(axis=0)
        elif self.pooling == 'weighted':
            if weights is None:
                weights = np.ones(len(token_ids))
            weights = np.array(weights).reshape(-1, 1)
            return (embeddings * weights).sum(axis=0) / weights.sum()
        else:
            raise ValueError(f"Unknown pooling method: {self.pooling}")

    def transform(self, token_sequences: List[List[int]],
                  tfidf_weights: Optional[List[List[float]]] = None) -> np.ndarray:
        """
        Transform token sequences to embedding features

        Args:
            token_sequences: List of token ID sequences
            tfidf_weights: Optional TF-IDF weights for weighted pooling

        Returns:
            Feature matrix [n_samples, embed_dim] or [n_samples, pca_dims]
        """
        features = []
        for i, seq in enumerate(token_sequences):
            if tfidf_weights is not None:
                weights = tfidf_weights[i]
            else:
                weights = None
            features.append(self._pool_embeddings(seq, weights))

        X = np.array(features)

        # Apply PCA if requested
        if self.use_pca and self.pca_dims is not None:
            if self.pca is None:
                self.pca = PCA(n_components=self.pca_dims, random_state=42)
                X = self.pca.fit_transform(X)
                var_explained = self.pca.explained_variance_ratio_.sum()
                print(f"✓ Applied PCA: {self.embed_dim}D → {self.pca_dims}D")
                print(f"  Variance explained: {var_explained:.2%}")
            else:
                X = self.pca.transform(X)

        return X


# ==========================================
# 4. TF-IDF WEIGHTED EMBEDDINGS
# ==========================================

class TfidfWeightedEmbeddings:
    """
    Combine TF-IDF weights with word embeddings
    """

    def __init__(self, embedding_matrix: np.ndarray, vocab: Dict[str, int]):
        self.embedding_matrix = embedding_matrix
        self.vocab = vocab
        self.tfidf_vectorizer = None
        self.id_to_word = {v: k for k, v in vocab.items()}

    def fit_transform(self, texts: List[str], token_sequences: List[List[int]]) -> np.ndarray:
        """
        Compute TF-IDF weighted embeddings

        Args:
            texts: List of text strings
            token_sequences: Corresponding token ID sequences

        Returns:
            Weighted embedding features
        """
        # Compute TF-IDF
        self.tfidf_vectorizer = TfidfVectorizer(vocabulary=self.vocab)
        tfidf_matrix = self.tfidf_vectorizer.fit_transform(texts)

        # Extract TF-IDF weights for each sequence
        features = []
        for i, seq in enumerate(token_sequences):
            if len(seq) == 0:
                features.append(np.zeros(self.embedding_matrix.shape[1]))
                continue

            # Get TF-IDF weights for this document
            doc_tfidf = tfidf_matrix[i].toarray().flatten()

            # Weight embeddings by TF-IDF
            weighted_sum = np.zeros(self.embedding_matrix.shape[1])
            total_weight = 0

            for token_id in seq:
                word = self.id_to_word.get(token_id)
                if word and word in self.vocab:
                    tfidf_weight = doc_tfidf[self.vocab[word] - 1]
                    weighted_sum += self.embedding_matrix[token_id] * tfidf_weight
                    total_weight += tfidf_weight

            if total_weight > 0:
                features.append(weighted_sum / total_weight)
            else:
                features.append(weighted_sum)

        return np.array(features)


# ==========================================
# 5. COMPLETE PIPELINE
# ==========================================

class ArabicFeaturePipeline:
    """
    Complete feature extraction pipeline with multiple options
    """

    def __init__(self, method='embedding', **kwargs):
        """
        Args:
            method: 'statistical', 'embedding', 'tfidf_weighted', or 'combined'
            **kwargs: Parameters for specific methods
        """
        self.method = method
        self.kwargs = kwargs
        self.extractor = None

    def fit_transform(self, texts: List[str], token_sequences: List[List[int]],
                     labels: np.ndarray, embedding_matrix: Optional[np.ndarray] = None,
                     vocab: Optional[Dict[str, int]] = None) -> np.ndarray:
        """
        Extract features based on chosen method
        """
        if self.method == 'statistical':
            selector_method = self.kwargs.get('selector', 'chi2')
            k = self.kwargs.get('k', 500)
            self.extractor = ArabicFeatureSelector(method=selector_method, k=k)
            X = self.extractor.fit_transform(texts, labels)

            # Optional: apply dimensionality reduction
            if self.kwargs.get('reduce_dim', False):
                n_components = self.kwargs.get('n_components', 50)
                reducer = DimensionalityReducer(method='svd', n_components=n_components)
                X = reducer.fit_transform(X)

            return X

        elif self.method == 'embedding':
            if embedding_matrix is None or vocab is None:
                raise ValueError("embedding_matrix and vocab required for embedding method")

            pooling = self.kwargs.get('pooling', 'mean')
            use_pca = self.kwargs.get('use_pca', False)
            pca_dims = self.kwargs.get('pca_dims', 30)

            self.extractor = EmbeddingFeatureExtractor(
                embedding_matrix, vocab, pooling=pooling,
                use_pca=use_pca, pca_dims=pca_dims
            )
            X = self.extractor.transform(token_sequences)
            return X

        elif self.method == 'tfidf_weighted':
            if embedding_matrix is None or vocab is None:
                raise ValueError("embedding_matrix and vocab required")

            self.extractor = TfidfWeightedEmbeddings(embedding_matrix, vocab)
            X = self.extractor.fit_transform(texts, token_sequences)
            return X

        elif self.method == 'combined':
            # Combine statistical + embedding features
            stat_selector = ArabicFeatureSelector(method='chi2', k=200)
            X_stat = stat_selector.fit_transform(texts, labels)

            emb_extractor = EmbeddingFeatureExtractor(
                embedding_matrix, vocab, pooling='mean'
            )
            X_emb = emb_extractor.transform(token_sequences)

            # Concatenate features
            if hasattr(X_stat, 'toarray'):
                X_stat = X_stat.toarray()
            X = np.hstack([X_stat, X_emb])

            print(f"✓ Combined features: {X_stat.shape[1]} statistical + {X_emb.shape[1]} embedding = {X.shape[1]} total")
            return X

        else:
            raise ValueError(f"Unknown method: {self.method}")


# ==========================================
# 6. EXAMPLE USAGE
# ==========================================

def example_feature_selection(texts_train, texts_test, y_train, y_test,
                              final_list_train, final_list_test,
                              embedding_matrix, vocab):
    """
    Example showing different feature selection approaches
    """
    print("=" * 60)
    print("FEATURE SELECTION EXAMPLES")
    print("=" * 60)

    # Example 1: Chi-square feature selection
    print("\n1️⃣ Chi-square Feature Selection")
    print("-" * 60)
    selector = ArabicFeatureSelector(method='chi2', k=300)
    X_train_chi2 = selector.fit_transform(texts_train, y_train)
    X_test_chi2 = selector.transform(texts_test)
    print(f"Train shape: {X_train_chi2.shape}")
    print(f"Test shape: {X_test_chi2.shape}")

    # Example 2: Mutual Information
    print("\n2️⃣ Mutual Information Feature Selection")
    print("-" * 60)
    selector_mi = ArabicFeatureSelector(method='mutual_info', k=300)
    X_train_mi = selector_mi.fit_transform(texts_train, y_train)
    X_test_mi = selector_mi.transform(texts_test)

    # Example 3: Mean pooling of embeddings
    print("\n3️⃣ Mean Pooling of Word Embeddings")
    print("-" * 60)
    emb_extractor = EmbeddingFeatureExtractor(
        embedding_matrix, vocab, pooling='mean'
    )
    X_train_emb = emb_extractor.transform(final_list_train)
    X_test_emb = emb_extractor.transform(final_list_test)
    print(f"Train shape: {X_train_emb.shape}")
    print(f"Test shape: {X_test_emb.shape}")

    # Example 4: Embeddings with PCA
    print("\n4️⃣ Embeddings + PCA")
    print("-" * 60)
    emb_pca = EmbeddingFeatureExtractor(
        embedding_matrix, vocab, pooling='mean',
        use_pca=True, pca_dims=30
    )
    X_train_pca = emb_pca.transform(final_list_train)
    X_test_pca = emb_pca.transform(final_list_test)
    print(f"Train shape: {X_train_pca.shape}")

    # Example 5: TF-IDF weighted embeddings
    print("\n5️⃣ TF-IDF Weighted Embeddings")
    print("-" * 60)
    tfidf_emb = TfidfWeightedEmbeddings(embedding_matrix, vocab)
    X_train_tfidf = tfidf_emb.fit_transform(texts_train, final_list_train)
    print(f"Train shape: {X_train_tfidf.shape}")

    print("\n" + "=" * 60)
    print("All features extracted successfully! ✓")
    print("=" * 60)

    return {
        'chi2': (X_train_chi2, X_test_chi2),
        'mutual_info': (X_train_mi, X_test_mi),
        'embeddings': (X_train_emb, X_test_emb),
        'embeddings_pca': (X_train_pca, X_test_pca),
        'tfidf_weighted': (X_train_tfidf, None)
    }

# FULL PIPILINE